# Beginner tour 3: How the model learns

In notebook 2 the line `self.model.train(...)` taught a model, and the model then
**guessed** the score of every undocked molecule. This notebook opens that box:

1. What "learning" means: adjusting numbers to make fewer mistakes
2. **PyTorch**, the toolbox the project uses, from zero
3. Training on real embeddings, and testing on molecules the model never saw
4. The project's real model: a guess **and** an uncertainty
5. Which backbone gives the best guesses, and why that matters
6. Reading the project's real training code

No maths beyond "smaller error is better". Run cells top to bottom with **Shift + Enter**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tour import *

scores, embeddings = load_mini()
y = -scores["score"].to_numpy(np.float32)       # flip the sign: from now on HIGHER = better
print(len(y), "molecules; best (flipped) score", y.max())

🐍 **Python corner: `.to_numpy()` and the minus sign.** `scores["score"]` is a table column,
and `.to_numpy()` turns it into a plain NumPy array. Putting `-` in front flips every number.
The project does exactly this (`self._sign = -1.0`), so that *a bigger guess = a better
molecule*. From here on, **higher y = better**.

---
## 1. What "learning" means

Start with the simplest possible model: guess the score from **one** number, the molecule's
size (its number of heavy atoms, from notebook 1).

> guess = **w** × size + **b**

`w` and `b` are the model's **parameters**: numbers we're free to choose. Learning means
**finding the values of w and b that make the guesses as close to the truth as possible.**

How close? We measure the **error**, here the *mean squared error* (MSE): for each molecule
take (guess − truth), square it so negatives don't cancel positives, and average. Smaller =
better. In machine learning this number is called the **loss**.

In [ ]:
from rdkit import Chem
size = np.array([Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in scores["smiles"]], dtype=np.float32)

def loss(w, b):
    guess = w * size + b
    return np.mean((guess - y) ** 2)

for w in [0.0, 0.1, 0.2, 0.3]:
    print(f"w = {w:.1f}, b = 4:  loss = {loss(w, 4.0):.2f}")

Try a whole range of `w` values and draw the loss. Learning is finding the bottom of this valley.

In [ ]:
ws = np.linspace(-0.1, 0.5, 61)
plt.figure(figsize=(5, 3))
plt.plot(ws, [loss(w, 4.0) for w in ws])
plt.xlabel("w"); plt.ylabel("loss (smaller = better)"); plt.title("b fixed at 4")
plt.show()

With 2 parameters we could try every combination. The project's models have **hundreds of
thousands** of parameters, so trying them all is impossible. Instead we use a trick called
**gradient descent**:

> stand somewhere in the valley, feel which way is downhill, take a small step, repeat.

"Which way is downhill" is the **gradient**. Computing it by hand is tedious, so we let
PyTorch do it.

---
## 2. PyTorch from zero

🐍 **Python corner: PyTorch tensors.** A **tensor** is PyTorch's version of a NumPy array: a
grid of numbers. The magic is `requires_grad=True`, which tells PyTorch to keep track of how
every result was computed, so it can work out gradients automatically.

In [ ]:
import torch

w = torch.tensor(0.0, requires_grad=True)     # start at w = 0, b = 0
b = torch.tensor(0.0, requires_grad=True)
x = torch.tensor((size - size.mean()) / size.std())   # rescale size to "around 0, spread 1": makes steps well-behaved
t = torch.tensor(y)

guess = w * x + b
L = ((guess - t) ** 2).mean()
L.backward()                                  # <- PyTorch computes the gradients
print("loss:", L.item())
print("gradient for w:", w.grad.item(), "  (negative: increasing w would lower the loss)")
print("gradient for b:", b.grad.item())

Now the loop: compute the loss, get gradients, take a small step **against** the gradient
(downhill), repeat. The step size is called the **learning rate**.

🐍 **Python corner: `with torch.no_grad():`** means "don't track these lines". Updating the
parameters themselves shouldn't count as part of the model's computation.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1
history = []

for step in range(50):
    L = ((w * x + b - t) ** 2).mean()          # 1. how wrong are we?
    L.backward()                               # 2. which way is downhill?
    with torch.no_grad():
        w -= learning_rate * w.grad            # 3. take a small step downhill
        b -= learning_rate * b.grad
        w.grad.zero_(); b.grad.zero_()         # 4. forget this step's gradients before the next one
    history.append(L.item())

plt.figure(figsize=(5, 3)); plt.plot(history); plt.xlabel("step"); plt.ylabel("loss"); plt.show()
print(f"learned: w = {w.item():.2f}, b = {b.item():.2f}, final loss {history[-1]:.2f}")

**That's all training is, at any size.** The project's big models do steps 1–4 thousands of
times, just with many more parameters.

The four steps are always the same, so PyTorch packages them. A **layer** like `nn.Linear`
holds the parameters, and an **optimizer** does step 3. Here's the standard pattern you'll
see in every PyTorch program, including the project's:

```python
optimizer.zero_grad()      # forget old gradients
loss = loss_fn(model(x), y)  # forward pass: guess, then measure
loss.backward()            # gradients
optimizer.step()           # update the parameters
```

---
## 3. Learning from real embeddings, and testing fairly

Size is one number. An embedding is **768** numbers (Uni-Mol2). A linear model now has 768
`w`s plus one `b`: one weight per embedding number.

**Testing fairly.** A model can memorize the molecules it was trained on. The only honest
test is on molecules it **never saw**. So we split:
* **train:** 420 random molecules (2% of the pool, about what active learning has after 2 rounds)
* **test:** the other 20,624

We judge the guesses with **Spearman correlation**: does the model put molecules in the right
*order*? 1 = perfect order, 0 = random. Order is what matters, because we only ever pick the
top of the list.

In [ ]:
import torch.nn as nn

rng = np.random.default_rng(0)
order = rng.permutation(len(y))
train, test = order[:420], order[420:]

def spearman(a, b):
    return pd.Series(a).corr(pd.Series(b), method="spearman")

X = torch.tensor(embeddings["unimol2"])
X = (X - X[train].mean(0)) / (X[train].std(0) + 1e-6)     # rescale each column using TRAIN statistics only
T = torch.tensor(y)

torch.manual_seed(0)
model = nn.Linear(768, 1)                                 # 768 weights + 1 bias
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-2)
print("number of parameters:", sum(p.numel() for p in model.parameters()))

for step in range(300):
    optimizer.zero_grad()
    guess = model(X[train]).squeeze(1)
    loss = ((guess - T[train]) ** 2).mean()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    guess_train = model(X[train]).squeeze(1).numpy()
    guess_test = model(X[test]).squeeze(1).numpy()
print(f"Spearman on TRAIN molecules: {spearman(guess_train, y[train]):.3f}")
print(f"Spearman on TEST molecules:  {spearman(guess_test, y[test]):.3f}   <- the honest number")

Two things to notice:
* The **train** number is better than the **test** number. The model knows its own training
  molecules better than new ones. If you only looked at train, you'd fool yourself.
* The **test** number is still high. From just 420 docked molecules, a simple model orders
  20,624 unseen ones quite well.

`weight_decay` gently pulls weights toward zero, which stops a model with 769 parameters from
memorizing 420 molecules. Try `weight_decay=0` and see what happens to the gap.

In [ ]:
plt.figure(figsize=(4.5, 4.5))
plt.scatter(y[test], guess_test, s=2, alpha=0.3)
plt.xlabel("true (flipped) score"); plt.ylabel("model's guess")
plt.title("Uni-Mol2 + a linear model, unseen molecules"); plt.show()

---
## 4. The project's real model: a guess AND an uncertainty

The project's basic surrogate, `SingleBackboneMVESurrogate` in `surrogates.py`, has the same
idea with two upgrades:

1. **A small neural network** instead of a straight line: layers of `Linear` with
   "bends" (`ReLU`) in between, so it can learn curved relationships.
2. **Two outputs per molecule:** `mu` (the guess) and `var` (how unsure it is). This is
   called **MVE**, *mean–variance estimation*.

Let's look inside. `print(model)` lists a PyTorch model's layers.

In [ ]:
import surrogates
net = surrogates._DualMVEModel(in_dim=768)
print(net)
print("\nnumber of parameters:", f"{sum(p.numel() for p in net.parameters()):,}")

How to read that:
* `backbone`: 768 numbers → 256 → 128. Each `Linear` mixes numbers, `ReLU` adds a bend,
  `BatchNorm` keeps numbers in a sensible range, and `Dropout` randomly switches some off
  during training (another guard against memorizing).
* `head1`, `head2`: **two** independent heads. Each turns the 128 numbers into a guess (`mu_net`)
  and an uncertainty (`var_net`). The model averages the two heads.

About 264,000 parameters, versus 769 for our straight line. Every one of them is adjusted by
exactly the four-step loop from section 2.

**How does it learn the uncertainty?** Its loss isn't plain squared error but the **MVE loss**:

> loss = ½ × [ log(var) + (guess − truth)² / var ]

Read the two parts: a big error is forgiven if the model said it was unsure (dividing by a
big `var`), but saying "unsure" always costs something (`log(var)`). So the model learns to
be unsure **only where it really makes mistakes**. Here's the project's real function on two
made-up molecules:

In [ ]:
from molpal.models.losses import mve_loss

truth = torch.tensor([5.0])
print("guess 5, confident  (var 0.1):", round(mve_loss(torch.tensor([5.0]), torch.tensor([0.1]), truth).item(), 2))
print("guess 8, confident  (var 0.1):", round(mve_loss(torch.tensor([8.0]), torch.tensor([0.1]), truth).item(), 2), " <- wrong AND confident: big penalty")
print("guess 8, unsure     (var 9.0):", round(mve_loss(torch.tensor([8.0]), torch.tensor([9.0]), truth).item(), 2), " <- wrong but said so: smaller penalty")
print("guess 5, unsure     (var 9.0):", round(mve_loss(torch.tensor([5.0]), torch.tensor([9.0]), truth).item(), 2), " <- right but unsure: still pays for it")

Don't worry that the first loss is negative. With this formula only *comparisons*
matter: lower is better, and "wrong and confident" is by far the worst.

(The project adds a small second term, a *Spearman loss*, that rewards getting the **order**
right. That's `CombinedLoss` in `molpal/models/losses.py`.)

Now train the real surrogate on the same 420 molecules. 🐍 `fit(X, y)` trains it and
`predict(X)` returns **two** arrays, which we unpack into `mu, sigma`. (`sigma` is the square
root of `var`, the typical size of the error.)

In [ ]:
torch.manual_seed(0)
X_uni = embeddings["unimol2"]
s = surrogates.SingleBackboneMVESurrogate(in_dim=768)
s.fit(X_uni[train], y[train], epochs=50)
mu, sigma = s.predict(X_uni[test])
print(f"Spearman on TEST molecules: {spearman(mu, y[test]):.3f}")

**Does the uncertainty mean anything?** If it does, the model's errors should be bigger
where it says it's unsure. Split the test molecules into 5 groups, from most confident to
least confident, and compare the average error:

In [ ]:
err = np.abs(mu - y[test])
groups = pd.qcut(sigma, 5, labels=["most confident", "2", "3", "4", "least confident"])
print(pd.Series(err).groupby(groups, observed=True).mean().round(2).to_string())

Yes: where the model says "unsure", its errors really are bigger. That's what makes
uncertainty useful for **deciding what to dock next**, which is notebook 4's topic.

---
## 5. Which backbone gives the best guesses?

Same model, same 420 training molecules, three different embeddings. Two measures:
* **Spearman** on the test molecules (order quality)
* **found**: of the true best 100, how many land in the model's top 1,260 guesses? (1,260
  is the budget from notebook 2, so this is roughly "how many would we find if we docked
  its top guesses".)

We also try the plain **linear** model on each, as a comparison.

🐍 **Python corner: building a table row by row.** We collect one dictionary per backbone in
a list, then `pd.DataFrame(rows)` turns the list into a table.

In [ ]:
best100 = set(np.argsort(-y)[:100].tolist())
rows = []
for name in BACKBONES:
    Xb = embeddings[name]
    torch.manual_seed(0)
    s = surrogates.SingleBackboneMVESurrogate(in_dim=Xb.shape[1])
    s.fit(Xb[train], y[train], epochs=50)
    mu_b, _ = s.predict(Xb[test])
    found = len(best100 & set(test[np.argsort(-mu_b)[:1260]].tolist()))

    # the linear comparison (closed-form "ridge regression": a straight line with weight decay)
    A = np.c_[Xb[train], np.ones(len(train))]
    w_lin = np.linalg.solve(A.T @ A + 10 * np.eye(A.shape[1]), A.T @ y[train])
    lin = np.c_[Xb[test], np.ones(len(test))] @ w_lin

    rows.append({"backbone": name, "neural net Spearman": round(spearman(mu_b, y[test]), 3),
                 "linear Spearman": round(spearman(lin, y[test]), 3), "best-100 found": found})
pd.DataFrame(rows)

This table is the heart of the project's findings, in miniature:

* **Backbones differ enormously.** MoLFormer, which reads SMILES text, orders molecules far
  worse than the other two. Its "similar molecules" aren't similar in the way docking cares
  about. (You saw a hint of this with the nearest neighbours in notebook 1.)
* **A bigger model isn't automatically better.** On Uni-Mol2 a straight line does about as
  well as the neural net. On MHG-GED the neural net is much better. Which model fits depends
  on the embedding.
* That's why the project spent so much effort on **which embeddings** to use. Its write-up
  concludes that what the model can *see* mattered far more than clever tricks downstream
  (`al-molecular/docs/adaptive_acq_results.md`, §1).

One seed and one split, so treat small differences with care.

---
## 6. Reading the project's real training code

Here's the training function every MVE surrogate uses, `_train_model` in `surrogates.py`.
Look for the four steps from section 2.

In [ ]:
import inspect
src = inspect.getsource(surrogates._train_model)
print(src[src.index("    n = X.shape[0]"):])        # skip the long explanation at the top

| Line | What it does |
|---|---|
| `Xt = torch.tensor(X, ..., device=DEVICE)` | move the data to the GPU once (`DEVICE` is `cuda` if there's a GPU) |
| `opt = torch.optim.AdamW(...)` | the optimizer: a smarter version of "step downhill" |
| `for _ in range(epochs):` | an **epoch** = one pass over all training molecules |
| `perm = torch.randperm(n, ...)` and `for start in range(0, n, batch):` | shuffle, then go through the data in **mini-batches** of 256 molecules |
| `opt.zero_grad()` | step 4: forget old gradients |
| `mu, var = model(xb)` and `loss = loss_fn(mu, var, yb)` | step 1: guess and measure (MVE + Spearman loss) |
| `scaler.scale(loss).backward()` | step 2: gradients. The `scaler` is part of **mixed precision**, which uses smaller numbers (16-bit) to run faster on a GPU |
| `nn.utils.clip_grad_norm_(...)` | safety: if a gradient is huge, shrink it so one bad step can't wreck the model |
| `scaler.step(opt)` | step 3: update the parameters |

Same four steps as our 10-line loop, plus speed and safety details.

**One more design choice: keep learning, or start fresh?** `SingleBackboneMVESurrogate` keeps
the **same** network from round to round, so each round continues from the last ("warm
start"). The project's current default model, `LTAllSurrogate`, deliberately builds **fresh**
networks every round. Both are reasonable, and it's an example of a decision you'd test with
an experiment.

In [ ]:
s = surrogates.SingleBackboneMVESurrogate(in_dim=768)
net_before = s._model
s.fit(X_uni[train[:200]], y[train[:200]], epochs=5)
s.fit(X_uni[train], y[train], epochs=5)              # "next round", more data
print("same network object after two rounds of fit()?", s._model is net_before)

---
## Summary

* **Learning** = adjusting parameters to lower a **loss** (an error score), by repeated small
  downhill steps (**gradient descent**).
* **PyTorch** computes the gradients for you. Every training loop is: `zero_grad` → guess and
  measure → `backward` → `step`.
* Always judge a model on molecules it **never saw** (train/test split).
* The project's surrogate is a small neural network that outputs a guess `mu` **and** an
  uncertainty `sigma`, trained with the MVE loss. Its uncertainty is meaningful: bigger
  where errors are bigger.
* **Which backbone** the embeddings come from matters enormously, more than which model sits on top.

🐍 **Python you met:** tensors, `requires_grad`, `.backward()`, `.grad`, `torch.no_grad()`,
`nn.Linear`, optimizers, `print(model)`, unpacking two return values, `pd.qcut`, building a
table from a list of dicts.

## Try it yourself
Answers, run on the same real data: [solutions/3_solutions.ipynb](solutions/3_solutions.ipynb). Try first!

1. In section 3, set `weight_decay=0`. What happens to the train vs test gap?
2. In section 3, use 100 training molecules instead of 420 (`order[:100]`). How much worse is the test Spearman?
3. In section 5, use `epochs=200`. Does the neural net improve, or start memorizing?

**Next:** `4_choosing_molecules.ipynb`: given guesses and uncertainties, which molecules
should we dock? And how can we combine several backbones?